In [4]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool
from langgraph.types import Command

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
documents = [
    "LangGraph is a library for building stateful multi-agent applications using graphs.",
    "RAG stands for Retrieval Augmented Generation. It combines retrieval with LLM generation.",
    "The Eiffel Tower is located in Paris, France and was built in 1889.",
    "Vector databases store embeddings for semantic similarity search.",
    "Python is a high-level programming language known for its simplicity.",
    "CRAG improves RAG by grading retrieved documents and rewriting queries when retrieval fails.",
]
grade_system_prompt="You are answer evaluator of question.You checked the answer provided by the process is relevant or not. If relevent set the value of is_relevant to True"
class CRAGState(TypedDict):
    question:      str
    documents:     List[str]
    generation:    str
    rewrite_count: int

def retrieve_node(state: CRAGState):
    question = state["question"].lower()
    # keep any doc that shares a word with the question
    relevant = [d for d in documents if any(w in d.lower() for w in question.split())]
    return {"documents": relevant if relevant else documents[:2]}  # fallback

class GradeOutput(BaseModel):
    is_relevant: bool

def grade_node(state: CRAGState):
    # for each doc, ask LLM: does this doc help answer the question?
# collect only relevant ones → {"documents": relevant_docs}
# if none relevant → {"documents": []}
    answers=state["documents"]
    question=state["question"]
    struc_res = llm_openai.with_structured_output(GradeOutput)
    relevant_docs = []
    for doc in state["documents"]:
        response = struc_res.invoke([
            SystemMessage(content=grade_system_prompt),
            HumanMessage(content=f"Question: {question}\nDocument: {doc}")
        ])
        if response.is_relevant:
            relevant_docs.append(doc)

    return {"documents": relevant_docs}

def rewrite_node (state: CRAGState):
    # prompt: "Rewrite this question to improve document retrieval. Make it more specific."
# returns {"question": rewritten, "rewrite_count": state["rewrite_count"] + 1}
    question=state["question"]
    response = llm_openai.invoke([
        SystemMessage(content="Rewrite this question to improve document retrieval. Make it more specific and add relevant keywords."),
        HumanMessage(content=question)
    ])
    return {"question": response.content, "rewrite_count": state["rewrite_count"] + 1}

def generate_node (state: CRAGState):
    # prompt: "Answer the question using these documents: {docs}"
    docs_text = '\n\n'.join(state["documents"])
    question=state["question"]
    response = llm_openai.invoke([
        SystemMessage(content=f"Answer the question using only these documents:\n\n{docs_text}"),
        HumanMessage(content=question)
    ])
    return {"generation": response.content}

def route_after_grade(state: CRAGState) -> str:
    if len(state["documents"]) > 0:
        return "generate"
    if state["rewrite_count"] >= 2:
        return "generate"   # safety — generate even with no docs after 2 rewrites
    return "rewrite"
g_graph=StateGraph(CRAGState)
g_graph.add_node('retrieve_node',retrieve_node)
g_graph.add_node('grade_node',grade_node)
g_graph.add_node('rewrite_node',rewrite_node)
g_graph.add_node('generate_node',generate_node)
g_graph.add_edge(START,'retrieve_node')
g_graph.add_edge('retrieve_node','grade_node')
g_graph.add_conditional_edges('grade_node',route_after_grade,{"generate":'generate_node','rewrite':'rewrite_node'})
g_graph.add_edge('rewrite_node','retrieve_node')
g_graph.add_edge('generate_node',END)

app=g_graph.compile()

q1 = "What is RAG and how does it work?"        # good retrieval expected
q2 = "Who invented the telephone?"
result = app.invoke({
    "question": q1,
    "documents": [],
    "generation": "",
    "rewrite_count": 0
})
print(f"Question: {result['question']}")        # may be rewritten
print(f"Rewrites: {result['rewrite_count']}")
print(f"Docs used: {len(result['documents'])}")
print(f"Answer: {result['generation'][:150]}")


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeOutput(is_relevant=False), input_type=GradeOutput])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeOutput(is_relevant=True), input_type=GradeOutput])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeOutput(is_

Question: What is RAG and how does it work?
Rewrites: 0
Docs used: 1
Answer: RAG stands for **Retrieval Augmented Generation**. It works by **combining retrieval with LLM (large language model) generation**—the system first ret


In [7]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool
from langgraph.types import Command

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
documents = [
    "LangGraph is a library for building stateful multi-agent applications using graphs.",
    "RAG stands for Retrieval Augmented Generation. It combines retrieval with LLM generation.",
    "The Eiffel Tower is located in Paris, France and was built in 1889.",
    "Vector databases store embeddings for semantic similarity search.",
    "Python is a high-level programming language known for its simplicity.",
    "CRAG improves RAG by grading retrieved documents and rewriting queries when retrieval fails.",
]
grade_system_prompt="You are answer evaluator of question.You checked the answer provided by the process is relevant or not. If relevent set the value of is_relevant to True"

dec_system_prompt="Decide if this question needs document retrieval.Answer 'false' for general knowledge questions, 'true' for specific/factual questions."
ans_system_prompt="Check if this answer is grounded in the provided documents.Return false if the answer contains claims not supported by the documents."

class SelfRAGState(TypedDict):
    question:          str
    documents:         List[str]
    generation:        str
    rewrite_count:     int
    generation_count:  int
    should_retrieve: bool
    is_grounded: bool

class RetrieveDecision(BaseModel):
    should_retrieve: bool

class AnswerGrade(BaseModel):
    is_grounded: bool

def decide_node(state:SelfRAGState):
    question=state['question']
    llm_struc=llm_openai.with_structured_output(RetrieveDecision)
    is_ground=llm_struc.invoke([SystemMessage(content=dec_system_prompt),HumanMessage(content=f"{question}")]).should_retrieve
    return {"should_retrieve": is_ground}

def grade_answer_node(state:SelfRAGState):
    documents='\n'.join(state["documents"])
    receives=state['generation']+documents
    llm_struc=llm_openai.with_structured_output(AnswerGrade)
    is_ground=llm_struc.invoke([SystemMessage(content=ans_system_prompt),HumanMessage(content=f"{receives}")]).is_grounded
    return {"is_grounded": is_ground}

def route_after_answer_grade(state: SelfRAGState) -> str:
    if state["is_grounded"] or state["generation_count"] >= 2:
        return "end"
    return "generate"


def retrieve_node(state: SelfRAGState):
    question = state["question"].lower()
    # keep any doc that shares a word with the question
    relevant = [d for d in documents if any(w in d.lower() for w in question.split())]
    return {"documents": relevant if relevant else documents[:2]}  # fallback

class GradeOutput(BaseModel):
    is_relevant: bool

def grade_node(state: SelfRAGState):
    # for each doc, ask LLM: does this doc help answer the question?
# collect only relevant ones → {"documents": relevant_docs}
# if none relevant → {"documents": []}
    answers=state["documents"]
    question=state["question"]
    struc_res = llm_openai.with_structured_output(GradeOutput)
    relevant_docs = []
    for doc in state["documents"]:
        response = struc_res.invoke([
            SystemMessage(content=grade_system_prompt),
            HumanMessage(content=f"Question: {question}\nDocument: {doc}")
        ])
        if response.is_relevant:
            relevant_docs.append(doc)

    return {"documents": relevant_docs}

def rewrite_node (state: SelfRAGState):
    # prompt: "Rewrite this question to improve document retrieval. Make it more specific."
# returns {"question": rewritten, "rewrite_count": state["rewrite_count"] + 1}
    question=state["question"]
    response = llm_openai.invoke([
        SystemMessage(content="Rewrite this question to improve document retrieval. Make it more specific and add relevant keywords."),
        HumanMessage(content=question)
    ])
    return {"question": response.content, "rewrite_count": state["rewrite_count"] + 1}

def generate_node (state: SelfRAGState):
    # prompt: "Answer the question using these documents: {docs}"
    docs_text = '\n\n'.join(state["documents"])
    question=state["question"]
    if docs_text:
        system = f"Answer the question using only these documents:\n\n{docs_text}"
    else:
        system = "Answer the question using your knowledge."


    response = llm_openai.invoke([SystemMessage(content=system), HumanMessage(content=question)])
    return {"generation": response.content,"generation_count": state["generation_count"] + 1}

def route_after_grade(state: SelfRAGState) -> str:
    if len(state["documents"]) > 0:
        return "generate"
    if state["rewrite_count"] >= 2:
        return "generate"   # safety — generate even with no docs after 2 rewrites
    return "rewrite"

g_graph=StateGraph(SelfRAGState)
g_graph.add_node('decide_node',decide_node)
g_graph.add_node('retrieve_node',retrieve_node)
g_graph.add_node('grade_node',grade_node)
g_graph.add_node('rewrite_node',rewrite_node)
g_graph.add_node('generate_node',generate_node)
g_graph.add_node('grade_answer_node',grade_answer_node)
g_graph.add_edge(START,'decide_node')
g_graph.add_conditional_edges('decide_node',lambda s: "retrieve" if s["should_retrieve"] else "skip",{"retrieve":'retrieve_node','skip':'generate_node'})
g_graph.add_edge('retrieve_node','grade_node')
g_graph.add_conditional_edges('grade_node',route_after_grade,({"generate":'generate_node','rewrite':'rewrite_node'}))
g_graph.add_edge('rewrite_node','retrieve_node')
g_graph.add_edge('generate_node','grade_answer_node')
g_graph.add_conditional_edges('grade_answer_node',route_after_answer_grade,{"end":END,"generate":'generate_node'})

app=g_graph.compile()

r1 = app.invoke({"question": "What is RAG?", "documents": [], "generation": "",
                 "rewrite_count": 0, "generation_count": 0, "is_grounded": False})

# Should NOT retrieve
r2 = app.invoke({"question": "What is 10 + 5?", "documents": [], "generation": "",
                 "rewrite_count": 0, "generation_count": 0, "is_grounded": False})

for r, q in [(r1, "What is Rag?"), (r2, "What is 10+5?")]:
    print(f"Q: {q}")
    print(f"  Docs retrieved: {len(r['documents'])}")
    print(f"  Grounded: {r['is_grounded']}")
    print(f"  Answer: {r['generation'][:100]}\n")


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RetrieveDecision(should_retrieve=False), input_type=RetrieveDecision])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AnswerGrade(is_grounded=False), input_type=AnswerGrade])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=

Q: What is Rag?
  Docs retrieved: 0
  Grounded: False
  Answer: **RAG = Retrieval‑Augmented Generation**

In the context of modern AI and large language models (LLM

Q: What is 10+5?
  Docs retrieved: 0
  Grounded: False
  Answer: 15.



C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AnswerGrade(is_grounded=False), input_type=AnswerGrade])
  return self.__pydantic_serializer__.to_python(


In [1]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool
from langgraph.types import Command

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
documents = [
    "LangGraph is a library for building stateful multi-agent applications using graphs.",
    "RAG stands for Retrieval Augmented Generation. It combines retrieval with LLM generation.",
    "The Eiffel Tower is located in Paris, France and was built in 1889.",
    "Vector databases store embeddings for semantic similarity search.",
    "Python is a high-level programming language known for its simplicity.",
    "CRAG improves RAG by grading retrieved documents and rewriting queries when retrieval fails.",
]
grade_system_prompt="You are answer evaluator of question.You checked the answer provided by the process is relevant or not. If relevent set the value of is_relevant to True"

dec_system_prompt="Decide if this question needs document retrieval.Answer 'false' for general knowledge questions, 'true' for specific/factual questions."
ans_system_prompt="Check if this answer is grounded in the provided documents.Return false if the answer contains claims not supported by the documents."

class SelfRAGState(TypedDict):
    question:          str
    documents:         List[str]
    generation:        str
    rewrite_count:     int
    generation_count:  int
    should_retrieve: bool
    is_grounded: bool

class RetrieveDecision(BaseModel):
    should_retrieve: bool

class AnswerGrade(BaseModel):
    is_grounded: bool

class EvalScore(BaseModel):
    score: int    # 1-5
    reason: str

context_system_prompt="As context evaluator check the question and documents provided by user. create the score in range 1-5 based on the relevant where 1 is low and 5 is high. Whatever score you provide must come with justification. the response will be score in integer and reason in string format"

faith_system_prompt="As faithfull evaluator check the question and documents  and answer provided by user. create the score in range 1-5 based on the relevant where 1 is low and 5 is high. Whatever score you provide must come with justification. the response will be score in integer and reason in string format"

ans_system_prompt="As answer evaluator check the question and answer provided by user. create the score in range 1-5 based on the relevant where 1 is low and 5 is high. Whatever score you provide must come with justification. the response will be score in integer and reason in string format"


def evaluate_rag(question: str, documents: List[str], answer: str) -> dict:
    docs_text = '\n'.join(documents)

    res_struc=llm_openai.with_structured_output(EvalScore)
    cr= res_struc.invoke([SystemMessage(content=context_system_prompt),HumanMessage(content=f'question:{question} documents: {docs_text}')])
    f=res_struc.invoke([SystemMessage(content=faith_system_prompt),HumanMessage(content=f'question:{question} documents: {docs_text} answer: {answer}')])
    ar=res_struc.invoke([SystemMessage(content=ans_system_prompt),HumanMessage(content=f'question:{question} answer: {answer}')])


    # Judge 1: Context Relevance
    # Judge 2: Faithfulness
    # Judge 3: Answer Relevance
    # Each uses llm_openai.with_structured_output(EvalScore).invoke([...])

    return {
        "context_relevance": cr.score,
        "faithfulness":       f.score,
        "answer_relevance":   ar.score,
        "reasons": {
            "context": cr.reason,
            "faithful": f.reason,
            "answer": ar.reason
        }
    }

def decide_node(state:SelfRAGState):
    question=state['question']
    llm_struc=llm_openai.with_structured_output(RetrieveDecision)
    is_ground=llm_struc.invoke([SystemMessage(content=dec_system_prompt),HumanMessage(content=f"{question}")]).should_retrieve
    return {"should_retrieve": is_ground}

def grade_answer_node(state:SelfRAGState):
    documents='\n'.join(state["documents"])
    receives=state['generation']+documents
    llm_struc=llm_openai.with_structured_output(AnswerGrade)
    is_ground=llm_struc.invoke([SystemMessage(content=ans_system_prompt),HumanMessage(content=f"{receives}")]).is_grounded
    return {"is_grounded": is_ground}

def route_after_answer_grade(state: SelfRAGState) -> str:
    if state["is_grounded"] or state["generation_count"] >= 2:
        return "end"
    return "generate"


def retrieve_node(state: SelfRAGState):
    question = state["question"].lower()
    # keep any doc that shares a word with the question
    relevant = [d for d in documents if any(w in d.lower() for w in question.split())]
    return {"documents": relevant if relevant else documents[:2]}  # fallback

class GradeOutput(BaseModel):
    is_relevant: bool

def grade_node(state: SelfRAGState):
    # for each doc, ask LLM: does this doc help answer the question?
# collect only relevant ones → {"documents": relevant_docs}
# if none relevant → {"documents": []}
    answers=state["documents"]
    question=state["question"]
    struc_res = llm_openai.with_structured_output(GradeOutput)
    relevant_docs = []
    for doc in state["documents"]:
        response = struc_res.invoke([
            SystemMessage(content=grade_system_prompt),
            HumanMessage(content=f"Question: {question}\nDocument: {doc}")
        ])
        if response.is_relevant:
            relevant_docs.append(doc)

    return {"documents": relevant_docs}

def rewrite_node (state: SelfRAGState):
    # prompt: "Rewrite this question to improve document retrieval. Make it more specific."
# returns {"question": rewritten, "rewrite_count": state["rewrite_count"] + 1}
    question=state["question"]
    response = llm_openai.invoke([
        SystemMessage(content="Rewrite this question to improve document retrieval. Make it more specific and add relevant keywords."),
        HumanMessage(content=question)
    ])
    return {"question": response.content, "rewrite_count": state["rewrite_count"] + 1}

def generate_node (state: SelfRAGState):
    # prompt: "Answer the question using these documents: {docs}"
    docs_text = '\n\n'.join(state["documents"])
    question=state["question"]
    if docs_text:
        system = f"Answer the question using only these documents:\n\n{docs_text}"
    else:
        system = "Answer the question using your knowledge."


    response = llm_openai.invoke([SystemMessage(content=system), HumanMessage(content=question)])
    return {"generation": response.content,"generation_count": state["generation_count"] + 1}

def route_after_grade(state: SelfRAGState) -> str:
    if len(state["documents"]) > 0:
        return "generate"
    if state["rewrite_count"] >= 2:
        return "generate"   # safety — generate even with no docs after 2 rewrites
    return "rewrite"

g_graph=StateGraph(SelfRAGState)
g_graph.add_node('decide_node',decide_node)
g_graph.add_node('retrieve_node',retrieve_node)
g_graph.add_node('grade_node',grade_node)
g_graph.add_node('rewrite_node',rewrite_node)
g_graph.add_node('generate_node',generate_node)
g_graph.add_node('grade_answer_node',grade_answer_node)
g_graph.add_edge(START,'decide_node')
g_graph.add_conditional_edges('decide_node',lambda s: "retrieve" if s["should_retrieve"] else "skip",{"retrieve":'retrieve_node','skip':'generate_node'})
g_graph.add_edge('retrieve_node','grade_node')
g_graph.add_conditional_edges('grade_node',route_after_grade,({"generate":'generate_node','rewrite':'rewrite_node'}))
g_graph.add_edge('rewrite_node','retrieve_node')
g_graph.add_edge('generate_node','grade_answer_node')
g_graph.add_conditional_edges('grade_answer_node',route_after_answer_grade,{"end":END,"generate":'generate_node'})

#app=g_graph.compile()

good = evaluate_rag(
    question="What is RAG?",
    documents=["RAG stands for Retrieval Augmented Generation. It combines retrieval with LLM generation."],
    answer="RAG stands for Retrieval Augmented Generation, combining document retrieval with LLM generation."
)

# Bad RAG result (hallucinated answer)
bad = evaluate_rag(
    question="What is RAG?",
    documents=["RAG stands for Retrieval Augmented Generation. It combines retrieval with LLM generation."],
    answer="RAG was invented in 2019 at Google and has 100 million users worldwide."
)

for label, result in [("GOOD", good), ("BAD", bad)]:
    print(f"\n{label}:")
    print(f"  Context Relevance: {result['context_relevance']}/5")
    print(f"  Faithfulness:      {result['faithfulness']}/5")
    print(f"  Answer Relevance:  {result['answer_relevance']}/5")

C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvalScore(score=5, reason... answers the question.'), input_type=EvalScore])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvalScore(score=5, reason...ddresses the question.'), input_type=EvalScore])
  return self.__pydantic_serializer__.to_python(
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_n


GOOD:
  Context Relevance: 5/5
  Faithfulness:      5/5
  Answer Relevance:  5/5

BAD:
  Context Relevance: 5/5
  Faithfulness:      1/5
  Answer Relevance:  1/5


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvalScore(score=1, reason... the asked definition.'), input_type=EvalScore])
  return self.__pydantic_serializer__.to_python(


In [3]:
from langgraph.checkpoint.memory import MemorySaver
import operator
import os


from dotenv import load_dotenv
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import Literal, TypedDict, Annotated, List
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from langchain.tools import tool
from langgraph.types import Command

load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
documents = [
    "LangGraph is a library for building stateful multi-agent applications using graphs.",
    "RAG stands for Retrieval Augmented Generation. It combines retrieval with LLM generation.",
    "The Eiffel Tower is located in Paris, France and was built in 1889.",
    "Vector databases store embeddings for semantic similarity search.",
    "Python is a high-level programming language known for its simplicity.",
    "CRAG improves RAG by grading retrieved documents and rewriting queries when retrieval fails.",
]
grade_system_prompt="You are answer evaluator of question.You checked the answer provided by the process is relevant or not. If relevent set the value of is_relevant to True"

dec_system_prompt="Decide if this question needs document retrieval.Answer 'false' for general knowledge questions, 'true' for specific/factual questions."
ans_system_prompt="Check if this answer is grounded in the provided documents.Return false if the answer contains claims not supported by the documents."

class SelfRAGState(TypedDict):
    question:          str
    documents:         List[str]
    generation:        str
    rewrite_count:     int
    generation_count:  int
    should_retrieve: bool
    is_grounded: bool

class RetrieveDecision(BaseModel):
    should_retrieve: bool

class AnswerGrade(BaseModel):
    is_grounded: bool

def decide_node(state:SelfRAGState):
    question=state['question']
    llm_struc=llm_openai.with_structured_output(RetrieveDecision)
    is_ground=llm_struc.invoke([SystemMessage(content=dec_system_prompt),HumanMessage(content=f"{question}")]).should_retrieve
    return {"should_retrieve": is_ground}

def grade_answer_node(state:SelfRAGState):
    documents='\n'.join(state["documents"])
    receives=state['generation']+documents
    llm_struc=llm_openai.with_structured_output(AnswerGrade)
    is_ground=llm_struc.invoke([SystemMessage(content=ans_system_prompt),HumanMessage(content=f"{receives}")]).is_grounded
    return {"is_grounded": is_ground}

def route_after_answer_grade(state: SelfRAGState) -> str:
    if state["is_grounded"] or state["generation_count"] >= 2:
        return "end"
    return "generate"


def retrieve_node(state: SelfRAGState):
    question = state["question"].lower()
    # keep any doc that shares a word with the question
    relevant = [d for d in documents if any(w in d.lower() for w in question.split())]
    return {"documents": relevant if relevant else documents[:2]}  # fallback

class GradeOutput(BaseModel):
    is_relevant: bool

def grade_node(state: SelfRAGState):
    # for each doc, ask LLM: does this doc help answer the question?
# collect only relevant ones → {"documents": relevant_docs}
# if none relevant → {"documents": []}
    answers=state["documents"]
    question=state["question"]
    struc_res = llm_openai.with_structured_output(GradeOutput)
    relevant_docs = []
    for doc in state["documents"]:
        response = struc_res.invoke([
            SystemMessage(content=grade_system_prompt),
            HumanMessage(content=f"Question: {question}\nDocument: {doc}")
        ])
        if response.is_relevant:
            relevant_docs.append(doc)

    return {"documents": relevant_docs}

def rewrite_node (state: SelfRAGState):
    # prompt: "Rewrite this question to improve document retrieval. Make it more specific."
# returns {"question": rewritten, "rewrite_count": state["rewrite_count"] + 1}
    question=state["question"]
    response = llm_openai.invoke([
        SystemMessage(content="Rewrite this question to improve document retrieval. Make it more specific and add relevant keywords."),
        HumanMessage(content=question)
    ])
    return {"question": response.content, "rewrite_count": state["rewrite_count"] + 1}

def generate_node (state: SelfRAGState):
    # prompt: "Answer the question using these documents: {docs}"
    docs_text = '\n\n'.join(state["documents"])
    question=state["question"]
    if docs_text:
        system = f"Answer the question using only these documents:\n\n{docs_text}"
    else:
        system = "Answer the question using your knowledge."


    response = llm_openai.invoke([SystemMessage(content=system), HumanMessage(content=question)])
    return {"generation": response.content,"generation_count": state["generation_count"] + 1}

def route_after_grade(state: SelfRAGState) -> str:
    if len(state["documents"]) > 0:
        return "generate"
    if state["rewrite_count"] >= 2:
        return "generate"   # safety — generate even with no docs after 2 rewrites
    return "rewrite"

g_graph=StateGraph(SelfRAGState)
g_graph.add_node('decide_node',decide_node)
g_graph.add_node('retrieve_node',retrieve_node)
g_graph.add_node('grade_node',grade_node)
g_graph.add_node('rewrite_node',rewrite_node)
g_graph.add_node('generate_node',generate_node)
g_graph.add_node('grade_answer_node',grade_answer_node)
g_graph.add_edge(START,'decide_node')
g_graph.add_conditional_edges('decide_node',lambda s: "retrieve" if s["should_retrieve"] else "skip",{"retrieve":'retrieve_node','skip':'generate_node'})
g_graph.add_edge('retrieve_node','grade_node')
g_graph.add_conditional_edges('grade_node',route_after_grade,({"generate":'generate_node','rewrite':'rewrite_node'}))
g_graph.add_edge('rewrite_node','retrieve_node')
g_graph.add_edge('generate_node','grade_answer_node')
g_graph.add_conditional_edges('grade_answer_node',route_after_answer_grade,{"end":END,"generate":'generate_node'})

app=g_graph.compile()
# print("=== Node-level streaming ===")
# for chunk in app.stream(
#     {"question": "What is RAG?", "documents": [], "generation": "",
#      "rewrite_count": 0, "generation_count": 0,
#      "should_retrieve": False, "is_grounded": False}
# ):
#     for node_name in chunk:
#         print(f"  ✓ {node_name} completed")

print("\n=== Token streaming (generate_node only) ===")
for chunk, metadata in app.stream(
    {"question": "What is RAG?", "documents": [], "generation": "",
     "rewrite_count": 0, "generation_count": 0,
     "should_retrieve": False, "is_grounded": False},
    stream_mode="messages"
):
    if metadata.get("langgraph_node") == "generate_node":
        print(chunk.content, end="", flush=True)
print()


=== Token streaming (generate_node only) ===


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=RetrieveDecision(should_retrieve=False), input_type=RetrieveDecision])
  return self.__pydantic_serializer__.to_python(


**RAG** most commonly stands for **Retrieval‑Augmented Generation**, a hybrid AI architecture that combines two complementary components:

| Component | What it does | Why it helps |
|-----------|--------------|--------------|
| **Retriever** | Searches an external knowledge source (e.g., a vector store, database, or search engine) for passages that are relevant to the current prompt. | Gives the model up‑to‑date, factual, and domain‑specific information that it may not have memorized during pre‑training. |
| **Generator** | A large language model (LLM) that takes the retrieved text (often concatenated with the original query) and produces a natural‑language response. | Leverages the LLM’s fluency, reasoning, and language understanding to turn raw facts into coherent answers. |

### How RAG works (high‑level flow)

1. **Input query** → user asks a question or gives a task.  
2. **Embedding & similarity search** → the query is encoded (e.g., with a dense vector model) and matched agains

C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AnswerGrade(is_grounded=False), input_type=AnswerGrade])
  return self.__pydantic_serializer__.to_python(


**RAG = Retrieval‑Augmented Generation**

In the context of modern AI and large language models (LLMs), **RAG** stands for **Retrieval‑Augmented Generation**. It is a hybrid architecture that combines two complementary capabilities:

| Component | What it does | Typical model |
|-----------|--------------|---------------|
| **Retriever** | Searches an external knowledge source (e.g., a document corpus, vector database, search engine) to pull the most relevant pieces of information for a given query. | Dense‑vector retrievers (e.g., FAISS, ScaNN, Milvus), sparse retrievers (BM25), hybrid retrievers. |
| **Generator** | Takes the retrieved text (often called “documents” or “passages”) together with the original prompt and produces a fluent, context‑aware response. | Large language models such as GPT‑3/4, LLaMA, Claude, T5, etc. |

The overall workflow looks like this:

1. **User query** → sent to the **retriever**.  
2. Retriever returns *k* most relevant passages (k is usually 5‑20).  


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=AnswerGrade(is_grounded=False), input_type=AnswerGrade])
  return self.__pydantic_serializer__.to_python(
